In [2]:
import cv2
import time
import pandas as pd
import gradio as gr
from ultralytics import YOLO

In [3]:
model = YOLO("yolov8n.pt")

In [4]:
model = YOLO("yolov8n.pt")

In [5]:
def process_video(
    video_path,
    roi_x1,
    roi_y1,
    roi_x2,
    roi_y2,
    image_size=640,
    frame_skip=1
):
    if video_path is None:
        return None, None, "Please upload a video."

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        return None, None, "Could not open video."

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    video_fps = cap.get(cv2.CAP_PROP_FPS)

    if video_fps <= 0:
        video_fps = 30

    # ROI coordinates
    x1 = int(width * roi_x1 / 100)
    y1 = int(height * roi_y1 / 100)
    x2 = int(width * roi_x2 / 100)
    y2 = int(height * roi_y2 / 100)

    output_path = "processed_video.mp4"

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(
        output_path,
        fourcc,
        video_fps,
        (width, height)
    )

    previous_inside = set()
    all_ids = set()
    entered_ids = set()
    exited_ids = set()

    events = []

    max_roi_objects = 0

    frame_number = 0
    processed_frames = 0
    processing_start = time.time()

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_number += 1

        # Frame skipping
        if frame_number % frame_skip != 0:
            out.write(frame)
            continue

        processed_frames += 1

        # Resize frame for inference
        resized_frame = cv2.resize(
            frame,
            (image_size, image_size)
        )

        results = model.track(
            resized_frame,
            persist=True,
            tracker="bytetrack.yaml",
            verbose=False
        )

        current_inside = set()

        if results[0].boxes is not None:
            boxes = results[0].boxes
            for box in boxes:
                if box.id is None:
                    continue
                track_id = int(box.id[0])
                all_ids.add(track_id)

                # Coordinates from resized image
                bx1, by1, bx2, by2 = map(
                    int,
                    box.xyxy[0]
                )

                # Convert coordinates back to original frame
                bx1 = int(bx1 * width / image_size)
                bx2 = int(bx2 * width / image_size)
                by1 = int(by1 * height / image_size)
                by2 = int(by2 * height / image_size)

                center_x = int((bx1 + bx2) / 2)
                center_y = int((by1 + by2) / 2)

                class_id = int(box.cls[0])
                class_name = model.names[class_id]

                inside_roi = (
                    x1 <= center_x <= x2 and
                    y1 <= center_y <= y2
                )

                if inside_roi:
                    current_inside.add(track_id)

                # Bounding box
                cv2.rectangle(
                    frame,
                    (bx1, by1),
                    (bx2, by2),
                    (0, 255, 0),
                    2
                )

                # Tracking ID
                cv2.putText(
                    frame,
                    f"ID {track_id} - {class_name}",
                    (bx1, max(by1 - 10, 20)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (0, 255, 0),
                    2
                )

                # Center point
                cv2.circle(
                    frame,
                    (center_x, center_y),
                    4,
                    (0, 0, 255),
                    -1
                )

        # Entry detection
        entries = current_inside - previous_inside

        for track_id in entries:
            entered_ids.add(track_id)

            events.append({
                "Tracking ID": track_id,
                "Event Type": "Entry",
                "Timestamp": frame_number / video_fps
            })

        # Exit detection
        exits = previous_inside - current_inside

        for track_id in exits:
            exited_ids.add(track_id)

            events.append({
                "Tracking ID": track_id,
                "Event Type": "Exit",
                "Timestamp": frame_number / video_fps
            })

        previous_inside = current_inside.copy()

        max_roi_objects = max(
            max_roi_objects,
            len(current_inside)
        )

        # Draw ROI
        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            (255, 0, 0),
            3
        )

        cv2.putText(
            frame,
            "ROI",
            (x1, max(y1 - 10, 20)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 0, 0),
            2
        )

        # Current object count
        cv2.putText(
            frame,
            f"Current Objects: {len(current_inside)}",
            (20, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 255),
            2
        )

        # Unique object count
        cv2.putText(
            frame,
            f"Unique Objects: {len(all_ids)}",
            (20, 60),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 255),
            2
        )

        # FPS
        elapsed = time.time() - processing_start
        fps = processed_frames / elapsed if elapsed > 0 else 0

        cv2.putText(
            frame,
            f"FPS: {fps:.2f}",
            (20, 90),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 255),
            2
        )

        out.write(frame)

    cap.release()
    out.release()

    total_time = time.time() - processing_start
    average_fps = processed_frames / total_time if total_time > 0 else 0

    # Save events
    events_df = pd.DataFrame(
        events,
        columns=["Tracking ID", "Event Type", "Timestamp"]
    )

    events_path = "events.csv"
    events_df.to_csv(events_path, index=False)

    summary = (
        f"Total Objects: {len(all_ids)}\n"
        f"Total Entries: {len(entered_ids)}\n"
        f"Total Exits: {len(exited_ids)}\n"
        f"Maximum Objects in ROI: {max_roi_objects}\n"
        f"Average FPS: {average_fps:.2f}\n"
        f"Processing Time: {total_time:.2f} seconds"
    )

    return output_path, events_path, summary

In [13]:
videoo = [
    "videoo/video1.mp4",
    "videoo/video2.mp4",
    "videoo/video3.mp4"
]

for i, video in enumerate(videoo, 1):
    output, events, summary = process_video(
        video,
        20, 20, 80, 80,
        image_size=640,
        frame_skip=1
    )

    print(f"\nVideo {i}")
    print(summary)


Video 1
Total Objects: 37
Total Entries: 21
Total Exits: 20
Maximum Objects in ROI: 6
Average FPS: 2.30
Processing Time: 94.87 seconds

Video 2
Total Objects: 120
Total Entries: 75
Total Exits: 68
Maximum Objects in ROI: 13
Average FPS: 3.53
Processing Time: 111.41 seconds

Video 3
Total Objects: 62
Total Entries: 37
Total Exits: 33
Maximum Objects in ROI: 8
Average FPS: 3.72
Processing Time: 156.51 seconds


In [14]:
def performance_test(video_path):
    configurations = [
        ("640px", 640, 1),
        ("480px", 480, 1),
        ("640px + Frame Skipping", 640, 2)
    ]

    results = []

    for name, size, skip in configurations:
        start = time.time()

        _, _, summary = process_video(
            video_path,
            20, 20, 80, 80,
            image_size=size,
            frame_skip=skip
        )

        processing_time = time.time() - start

        fps_line = [
            line for line in summary.split("\n")
            if "Average FPS" in line
        ][0]

        fps = float(
            fps_line.split(":")[1].strip()
        )

        results.append({
            "Configuration": name,
            "Image Size": size,
            "Frame Skip": skip,
            "FPS": round(fps, 2),
            "Processing Time": round(processing_time, 2)
        })

    return pd.DataFrame(results)

In [16]:
performance_results = performance_test("videoo/video1.mp4")
performance_results

,Configuration,Image Size,Frame Skip,FPS,Processing Time
0,640px,640,1,2.33,93.86
1,480px,480,1,2.55,85.67
2,640px + Frame Skipping,640,2,1.93,56.54


In [17]:
def gradio_process(
    video,
    roi_x1,
    roi_y1,
    roi_x2,
    roi_y2,
    image_size,
    frame_skip
):

    return process_video(
        video,
        roi_x1,
        roi_y1,
        roi_x2,
        roi_y2,
        image_size,
        frame_skip
    )

In [18]:
with gr.Blocks() as app:
    gr.Markdown("# Smart Video Analytics System")

    video_input = gr.Video(
        label="Upload Video"
    )

    with gr.Row():
        roi_x1 = gr.Slider(
            0, 100, 20,
            label="ROI X1 (%)"
        )

        roi_y1 = gr.Slider(
            0, 100, 20,
            label="ROI Y1 (%)"
        )

    with gr.Row():
        roi_x2 = gr.Slider(
            0, 100, 80,
            label="ROI X2 (%)"
        )

        roi_y2 = gr.Slider(
            0, 100, 80,
            label="ROI Y2 (%)"
        )

    image_size = gr.Radio(
        choices=[640, 480],
        value=640,
        label="Image Size"
    )

    frame_skip = gr.Radio(
        choices=[1, 2],
        value=1,
        label="Frame Skipping"
    )

    process_button = gr.Button(
        "Start Processing"
    )

    output_video = gr.Video(
        label="Processed Video"
    )

    summary = gr.Textbox(
        label="Analytics Summary"
    )

    events_file = gr.File(
        label="Download events.csv"
    )

    process_button.click(
        fn=gradio_process,
        inputs=[
            video_input,
            roi_x1,
            roi_y1,
            roi_x2,
            roi_y2,
            image_size,
            frame_skip
        ],
        outputs=[
            output_video,
            events_file,
            summary
        ]
    )

In [19]:
app.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Exception in callback _ProactorBasePipeTransport._call_connection_lost()
handle: <Handle _ProactorBasePipeTransport._call_connection_lost()>
Traceback (most recent call last):
  File "C:\Users\Localws\AppData\Local\Programs\Python\Python313\Lib\asyncio\events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Localws\AppData\Local\Programs\Python\Python313\Lib\asyncio\proactor_events.py", line 165, in _call_connection_lost
    self._sock.shutdown(socket.SHUT_RDWR)
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^
ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host
Exception in callback _ProactorBasePipeTransport._call_connection_lost()
handle: <Handle _ProactorBasePipeTransport._call_connection_lost()>
Traceback (most recent call last):
  File "C:\Users\Localws\AppData\Local\Programs\Python\Python313\Lib\asyncio\events.py", line 89, in _run
    self._contex